# Exercise 2 — 1D Steady Heat Conduction, the FEM Way

**Companion to Lecture 2** of the lecture notes *Essential Concepts in FEM and Physics-Based ML*
(chapter “Exercise 2: Steady Heat Conduction”).

You will rebuild the *entire arc of Lecture 2* on a new physics: the steady temperature
field in a rod. Deliberately, the **intuitive picture comes last** — the point of this
exercise is to experience that the formal machinery works even before you have intuition.

**The setting.** A rod of length $L$, cross-section $A$, conductivity $\lambda$, insulated
on its lateral surface (heat flows only along $x$). A distributed heat source $s(x)$ (W/m)
acts along the rod. Unknown: the temperature field $T(x)$.

**How to work.** Fill in every `...` (TODO). Each part ends with a ✅ *check cell* —
run it; if it passes silently, move on. Don't peek ahead: the parts build on each other.

---
*Author: Ramses Sala · Copyright © 2025 Ramses Sala · Part of the course* Essential Concepts in FEM and Physics-Based ML. *Created with assistance of Claude Code for text, image and code refinement and generation.*

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

sp.init_printing()

# symbols used throughout
x, L, lam, A, s0, Tbar, h = sp.symbols('x L lambda A s_0 Tbar h', positive=True)


---
## Part 1 — Build the differential equation from physical relations

Below is a box of candidate relations. **Three of them are the ingredients you pick**
(one *kinematic*-type relation, one *material* law, one *balance* law). A fourth, the
trivial cross-section bundling D, we give you for free. The remaining two are distractors.

| id | relation | formula |
|----|----------|---------|
| A | temperature gradient | $g = \dfrac{dT}{dx}$ |
| B | Fourier's law | $q = -\lambda\, g$ |
| C | “conduction law” | $q = -\lambda\, T$ |
| D | heat-flow resultant | $Q = A\, q$ |
| E | steady heat balance of a slice | $\dfrac{dQ}{dx} = s(x)$ |
| F | “source law” | $\dfrac{dT}{dx} = s(x)$ |

Pick the letter for each role (strings, e.g. `role_kinematic = "A"`). Then, **on paper**,
draw the four-box diagram (the three-step diagram of Lecture 2, Section 2.2) and chain the
relations into one differential equation for $T(x)$.

In [ ]:
role_kinematic = ...   # 'A'..'F' : relates the primal field T to its gradient
role_material  = ...   # 'A'..'F' : the (experimentally measured) material law
role_balance   = ...   # 'A'..'F' : the conservation/balance statement


In [ ]:
# ✅ check — Part 1
picked = tuple(str(r).strip().upper() for r in (role_kinematic, role_material, role_balance))
assert picked == ('A', 'B', 'E'), \
    "Not quite. Which relation is pure geometry/definition? Which contains a material constant? Which balances in/out against the source?"
print("Correct. Chained together (with Q = A q):  d/dx( lambda*A * dT/dx ) + s(x) = 0")
print("Compare with the bar:   d/dx(   E   *A * du/dx ) + n(x) = 0   — same equation, new actors.")


**Question to answer in a sentence (markdown cell below):** Fourier's law carries a
minus sign, Hooke's law does not — yet the final ODE has the *same* sign pattern as the
bar's. Where did the minus sign go?

*(your answer here)*

---
## Part 2 — Boundary value problem and reference solution

**Our BVP** (constant $\lambda A$, constant source $s_0$):

$$\lambda A\, T'' + s_0 = 0 \quad\text{in } (0, L), \qquad T(0) = \bar T, \qquad Q(L) = 0 .$$

- Which condition is *essential* (Dirichlet), which *natural* (Neumann)?
- What does the homogeneous natural condition $Q(L)=0$ mean physically?
- Why would prescribing the flux at *both* ends be an ill-posed problem? (Think: rigid
  body mode of the spring chain. What is its thermal twin?)

**Task:** solve the BVP exactly with `sympy` (integrate twice, fit both conditions).
Note $Q = A q = -\lambda A\, T'$, so $Q(L)=0$ is simply $T'(L)=0$.

In [ ]:
T = sp.Function('T')

ode = sp.Eq(lam*A*T(x).diff(x, 2) + s0, 0)

# TODO: solve the ODE with the two boundary conditions T(0)=Tbar, T'(L)=0.
# hint: sp.dsolve(ode, T(x), ics={...}) -- fill in the two conditions;
#       T'(L) is written T(x).diff(x).subs(x, L)
T_exact = ...

T_exact = sp.simplify(T_exact.rhs if isinstance(T_exact, sp.Eq) else T_exact)
T_exact


In [ ]:
# ✅ check — Part 2
expected = Tbar + s0*(L*x - x**2/2)/(lam*A)
assert sp.simplify(T_exact - expected) == 0, "Integrate twice; use T(0)=Tbar and T'(L)=0 to fix both constants."
print("Exact solution confirmed:  T(x) = Tbar + s0*(L*x - x**2/2)/(lambda*A)")


---
## Part 3 — Collocation

Ansatz (one parameter):

$$\tilde T(x) = \bar T + a_1\, x\,(2L - x).$$

First convince yourself (in one line of algebra each) that this ansatz satisfies the
essential condition at $x=0$ **and** the natural condition at $x=L$, for *any* $a_1$.

Then compute the residual $R(x) = \lambda A\, \tilde T'' + s_0$ and collocate at $x = L/2$.

In [ ]:
a1 = sp.Symbol('a_1')
T_trial = Tbar + a1*x*(2*L - x)

# TODO: residual of the ODE for the trial function
R = ...

# TODO: solve R(L/2) = 0 for a1
a1_colloc = ...

a1_colloc


In [ ]:
# ✅ check — Part 3
assert sp.simplify(a1_colloc - s0/(2*lam*A)) == 0
assert sp.simplify(T_trial.subs(a1, a1_colloc) - T_exact) == 0
print("Collocation is EXACT here. Why? Look at the exact solution: it is itself of the")
print("form Tbar + a*x*(2L-x). The ansatz family contains the true solution —")
print("and then any sensible method finds it (lecture notes, Section 2.6).")


---
## Part 4 — Galerkin, and a case where the methods differ

Now a **linear source** $s(x) = s_0\, x / L$ (same BCs). The exact solution becomes a
cubic — *outside* our quadratic ansatz family. Now the methods must genuinely approximate.

1. Solve the BVP exactly again (as in Part 2).
2. Collocate the residual at $x=L/2$.
3. Do **Galerkin on the weak form**: with $\varphi_1 = x(2L-x)$,

$$K_{11} = \int_0^L \lambda A\, \varphi_1'^2\, dx, \qquad
  f_1 = \int_0^L \varphi_1\, s(x)\, dx, \qquad a_1 = f_1 / K_{11}.$$

(The boundary term of the weak form vanishes — why?)

In [ ]:
s_lin = s0*x/L

# TODO: exact solution for the linear source (same BCs)
T_exact_lin = ...

# TODO: collocation at L/2 for the linear source
a1_col_lin = ...

# TODO: Galerkin via the weak form
phi1 = x*(2*L - x)
K11 = ...
f1  = ...
a1_gal_lin = sp.simplify(f1/K11)

a1_col_lin, a1_gal_lin


In [ ]:
# ✅ check — Part 4
expected_T = Tbar + s0*(L*x/2 - x**3/(6*L))/(lam*A)
assert sp.simplify(T_exact_lin - expected_T) == 0, "Exact solution: integrate -s0*x/L twice, fit T(0)=Tbar, T'(L)=0."
assert sp.simplify(a1_col_lin - s0/(4*lam*A)) == 0, "Collocation: residual at x=L/2."
assert sp.simplify(a1_gal_lin - 5*s0/(16*lam*A)) == 0, "Galerkin: K11 = 4*lam*A*L^3/3, f1 = 5*s0*L^3/12."
print("OK. Collocation and Galerkin now give different answers — plot them:")

f_num = {L: 1, lam: 1, A: 1, s0: 1, Tbar: 0}
xx = np.linspace(0, 1, 200)
for expr, lab in [(T_exact_lin, 'exact (cubic)'),
                  ((Tbar + a1_col_lin*phi1), 'collocation @ L/2'),
                  ((Tbar + a1_gal_lin*phi1), 'Galerkin')]:
    fn = sp.lambdify(x, expr.subs(f_num))
    plt.plot(xx, fn(xx), label=lab)
plt.xlabel('x/L'); plt.ylabel('T - Tbar'); plt.legend(); plt.title('Linear source: one-parameter methods'); plt.show()


---
## Part 5 — The finite element: hat functions

Global polynomial ansatz functions got us this far, but inventing them for every new set
of boundary conditions is an art. Shape functions make it a *procedure*.

On one element of length $h$ with local coordinate $\xi \in [0,h]$:

$$N_1 = 1 - \xi/h, \qquad N_2 = \xi/h, \qquad
  \mathbf{B} = [N_1' \;\; N_2'] = \tfrac1h[-1 \;\; 1].$$

**Task:** compute the element conduction matrix and the consistent source vector

$$\mathbf{k}^e = \int_0^h \mathbf{B}^T \lambda A\, \mathbf{B}\, d\xi, \qquad
  \mathbf{f}^e = \int_0^h \mathbf{N}^T s_0\, d\xi .$$

In [ ]:
xi = sp.Symbol('xi', nonnegative=True)
N = sp.Matrix([[1 - xi/h, xi/h]])
B = N.diff(xi)

# TODO: element matrix and consistent load vector (sp.integrate works on matrices)
ke = ...
fe = ...

ke, fe


In [ ]:
# ✅ check — Part 5
ke_expected = (lam*A/h)*sp.Matrix([[1, -1], [-1, 1]])
fe_expected = (s0*h/2)*sp.Matrix([[1], [1]])
assert sp.simplify(ke - ke_expected) == sp.zeros(2, 2), "k_e should be lambda*A/h * [[1,-1],[-1,1]]"
assert sp.simplify(fe - fe_expected) == sp.zeros(2, 1), "f_e should be s0*h/2 * [1,1]^T"
print("The spring/bar element matrix — with lambda*A in place of E*A.")
print("You have just derived a thermal finite element.")


---
## Part 6 — Assemble and solve

Write a tiny FEM solver (numeric now — `numpy`): nodes at positions `xn` (any spacing!),
element $e$ between nodes $e$ and $e{+}1$.

* assemble $\mathbf{K}$ (add each $\mathbf{k}^e$ into the right slots) and $\mathbf{f}$
  (each element sends $s_0 h_e/2$ to both of its nodes);
* impose the essential BC $T_1 = \bar T$ by **partitioning** (lecture notes, eq. (1.5)):
  solve $\mathbf{K}_{ff} \mathbf{T}_f = \mathbf{f}_f - \mathbf{K}_{fp} \bar T$;
* the insulated end needs **no action** (natural BC!).

In [ ]:
def solve_heat_fem(xn, lamA, s0_val, Tbar_val):
    '''1D steady heat FEM. xn: array of node positions (any spacing).
    Returns nodal temperatures (incl. the prescribed one).'''
    n = len(xn)
    K = np.zeros((n, n))
    f = np.zeros(n)
    for e in range(n - 1):
        he = xn[e+1] - xn[e]
        ke = ...   # TODO: 2x2 element matrix as a numpy array
        fe = ...   # TODO: 2-vector consistent source
        # TODO: add ke into K at rows/cols [e, e+1], fe into f
        ...
    # essential BC: T[0] = Tbar_val, partition and solve
    Kff = K[1:, 1:]
    rhs = f[1:] - K[1:, 0]*Tbar_val
    Tf  = np.linalg.solve(Kff, rhs)
    return np.concatenate([[Tbar_val], Tf])

# four elements, deliberately non-uniform:
xn = np.array([0.0, 0.2, 0.45, 0.75, 1.0])
T_fem = solve_heat_fem(xn, lamA=1.0, s0_val=1.0, Tbar_val=0.0)
T_fem


In [ ]:
# ✅ check — Part 6   (constant source: FEM is EXACT at the nodes — lecture notes, Section 2.12)
T_nodes_exact = 0.0 + 1.0*(1.0*xn - xn**2/2)/1.0
assert np.allclose(T_fem, T_nodes_exact, atol=1e-12), \
    "Nodal values should match the exact solution to machine precision. Check assembly slots and the load vector."
print("Nodal exactness confirmed on a NON-uniform mesh. Plot the full picture:")

xx = np.linspace(0, 1, 300)
plt.plot(xx, 1.0*xx - xx**2/2, label='exact')
plt.plot(xn, T_fem, 'o-', label='FEM (4 elements, non-uniform)')
plt.xlabel('x'); plt.ylabel('T'); plt.legend(); plt.title('Constant source, insulated tip'); plt.show()


---
## Part 7 — Experiments (no checks; explore)

1. **Refine**: rerun with 2, 4, 8, 16 *uniform* elements for the **linear** source
   $s(x) = s_0 x/L$ (adapt the element load: a quick legal shortcut is to evaluate
   $s$ at the element midpoint and use $s(x_{\rm mid})\,h_e/2$ per node). Plot the error
   at $x=L$ against the element count. What do you observe?
2. **Second load case**: same mesh, add a prescribed heat input $P$ (W) at the right
   end. From the weak form's boundary term, this simply **adds $P$ to the load vector
   entry of the end node** — derive the sign from $[w\,\lambda A\,T']_0^L$ as a bonus.
   Note that $\mathbf{K}$ did not change: load cases reuse the same matrix.
3. **Composite rod**: make `lamA` element-dependent (a list) — a rod half copper, half
   steel. Where does the temperature gradient jump, and why is that physical?

In [ ]:
# your experiments here


---
## Part 8 — *Only now*: the intuitive picture

Each element you derived is a **thermal resistor**: it conducts the heat flow
$Q = -\frac{\lambda A}{h}\,(T_2 - T_1)$ between its two nodes — the heat-flow twin of the
spring $N = k(u_2 - u_1)$ (the extra minus is Fourier's), with conductance $G = \lambda A / h$
in place of stiffness.
Assembly is Kirchhoff's node balance; the essential BC is grounding a node to a fixed
temperature; a prescribed end heat input is a current source.

**Closing tasks (paper):**
1. For your 4-element mesh of Part 6, write the chain-of-resistors network and verify two
   entries of $\mathbf{K}$ against your assembled matrix.
2. In Lecture 1 the intuition (springs) came *first* and the mathematics confirmed it.
   Here the mathematics came first and intuition arrived last — yet you solved everything
   before Part 8 existed. Write three sentences: what does this tell you about how you
   will approach the *next* physics you have never seen (electrostatics, say)?